In [1]:
from Models.Unet import Unet
from torch.utils.data import DataLoader
from Transforms import (pad, trim_coils, combine_coil, toTensor, permute, 
                                   view_as_real, remove_slice_dim, fft_2d, normalize)
from Dataset.undersampled_dataset import undersampled_k_loader
from torchvision.transforms import Compose
import numpy as np
import torch

ModuleNotFoundError: No module named 'einops'

In [16]:
%load_ext autoreload
%autoreload 2 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
def collate_fn(data):
    undersampled, sampled = zip(*data)
    # slices = 0
    # for item in undersampled:
    #     slices += item.shape[0]

    # concat_undersampled = torch.zeros(((slices,) + tuple(undersampled[0].shape[2:])))
    # concat_sampled = torch.zeros(((slices,) + tuple(undersampled[0].shape[2:])))

    # current_index = 0

    # undersampled = [item.view((-1,) + item.shape[2:]) for item in undersampled] 
    # sampled = [item.view((-1,) + item.shape[2:]) for item in sampled] 

    return torch.concat(undersampled, dim=0), torch.concat(sampled, dim=0)


In [18]:
transforms = Compose(
    (
        trim_coils(12),
        pad((640, 320)), 
        fft_2d(axes=[2,3]),
        combine_coil(),
        normalize(),
        toTensor(),
        view_as_real(),
        permute() 
    )
)
dataset = undersampled_k_loader('D:/multicoil_train', transforms=transforms)
dataloader = DataLoader(dataset, batch_size=1, collate_fn=collate_fn)
    

In [19]:
model = Unet(2, 2)

In [20]:
sum([param.numel() for param in model.parameters()])

1939504

In [21]:
import cProfile
cProfile.run('next(iter(dataloader))', 'dataloader')

In [22]:
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), momentum=0.99, lr=0.0001)

In [23]:
def train(model, loss_function, optimizer, dataloader):
    cur_loss = 0
    for i, (undersampled, sampled) in enumerate(dataloader):

        optimizer.zero_grad()

        predicted_sampled = model(undersampled)
        loss = loss_function(predicted_sampled, sampled)
    
        loss.backward()
        optimizer.step()
        cur_loss += loss.item()
        if i % 10 == 9:
            print(f"Iteration: {i + 1}, Loss: {cur_loss:>7f}")
            cur_loss = 0


In [24]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), momentum=0.9, lr=0.001)

In [25]:
train(model, loss_fn, optimizer, dataloader)

Iteration: 10, Loss: 0.552391
Iteration: 20, Loss: 0.385253
Iteration: 30, Loss: 0.482085
Iteration: 40, Loss: 0.548485
Iteration: 50, Loss: 0.348545
Iteration: 60, Loss: 0.486772
Iteration: 70, Loss: 0.342552
Iteration: 80, Loss: 0.544297
Iteration: 90, Loss: 0.423155
Iteration: 100, Loss: 0.407481
Iteration: 110, Loss: 0.394631
Iteration: 120, Loss: 0.434917
Iteration: 130, Loss: 0.434872
Iteration: 140, Loss: 0.447616
Iteration: 150, Loss: 0.397403
Iteration: 160, Loss: 0.469087
Iteration: 170, Loss: 0.384657
Iteration: 180, Loss: 0.411418
Iteration: 190, Loss: 0.498358
Iteration: 200, Loss: 0.402734
Iteration: 210, Loss: 0.388431
Iteration: 220, Loss: 0.346450
Iteration: 230, Loss: 0.388270
Iteration: 240, Loss: 0.420417
Iteration: 250, Loss: 0.446334
Iteration: 260, Loss: 0.443070
Iteration: 270, Loss: 0.381680
Iteration: 280, Loss: 0.450967
Iteration: 290, Loss: 0.356449
Iteration: 300, Loss: 0.456284


OSError: Unable to open file (truncated file: eof = 20953088, sblock->base_addr = 0, stored_eof = 464502800)